# Thermal Real-ESRGAN — Colab training (160×120 → 640×480)

Fine-tunes **SRVGGNetCompact 64/16** (the architecture of `realesr-general-x4v3`) on thermal
imagery with sensor-specific degradation, then converts it to ncnn fp16 for the Android app.

**Before you start, put this on your Drive:**

```
MyDrive/thermal_sr/
├── raw/                 HR thermal images (FLIR ADAS v2 / KAIST / PBVS / anything ≥480px)
│   ├── flir_adas_v2/
│   └── kaist/
├── val_mag160/          ~20 real 160×120 captures from your camera (optional but recommended)
└── captures/            wall_300.npy — (N,H,W) static frames for FPN measurement (optional)
```

Runtime → Change runtime type → **T4 GPU** (or better).

Everything (checkpoints, logs, exports) is written to Drive, and every training cell uses
`--auto_resume`, so a disconnect costs you at most the iterations since the last checkpoint —
just re-run the cell.


## 1. GPU + Drive


In [ ]:
!nvidia-smi
from google.colab import drive
drive.mount('/content/drive')

import pathlib
ROOT = pathlib.Path('/content/drive/MyDrive/thermal_sr')
for sub in ('raw', 'val_mag160', 'captures', 'experiments', 'export'):
    (ROOT / sub).mkdir(parents=True, exist_ok=True)
print('drive root:', ROOT)
print('raw sources:', [p.name for p in (ROOT / 'raw').iterdir() if p.is_dir()])


## 2. Install Real-ESRGAN + this project's pipeline

Two things need patching before anything installs, both the same Python 3.13 issue:
`basicsr`'s and Real-ESRGAN's `setup.py` read their version with `exec()` and then
`locals()['__version__']`, which PEP 667 broke — you get `KeyError: '__version__'`.
So we patch basicsr's sdist before installing it, and skip Real-ESRGAN's `setup.py`
entirely by running the trainer from the repo directory instead of installing it.


In [ ]:
%cd /content
!rm -rf Real-ESRGAN MAG160_ThermalCam
!git clone -q https://github.com/xinntao/Real-ESRGAN.git
!git clone -q https://github.com/delphicchen/MAG160_ThermalCam.git
%cd /content/Real-ESRGAN
!cp -r /content/MAG160_ThermalCam/sr_train/options /content/MAG160_ThermalCam/sr_train/thermal_arch /content/MAG160_ThermalCam/sr_train/scripts .

# runtime deps (torch/torchvision/opencv/scipy ship with Colab)
!pip -q install addict yapf lmdb future tqdm onnx onnxsim ncnn pnnx tb-nightly

# --- basicsr ---------------------------------------------------------------
# basicsr's setup.py reads its version with exec() + locals()['__version__'], which
# PEP 667 broke on Python 3.13. pip cannot even *download* it with --no-binary,
# because that runs setup.py egg_info first — so fetch the sdist straight from PyPI,
# patch it, then install from the patched directory.
import json, pathlib, shutil, subprocess, sys, tarfile, urllib.request

work = pathlib.Path('/tmp/bs')
shutil.rmtree(work, ignore_errors=True)
work.mkdir(parents=True)

meta = json.load(urllib.request.urlopen('https://pypi.org/pypi/basicsr/1.4.2/json'))
url = next(u['url'] for u in meta['urls'] if u['packagetype'] == 'sdist')
tgz = work / 'basicsr.tar.gz'
urllib.request.urlretrieve(url, tgz)
with tarfile.open(tgz) as t:
    t.extractall(work, filter='data')
pkg = next(p for p in work.iterdir() if p.is_dir())

sp = pkg / 'setup.py'
src = sp.read_text()
before = src
src = src.replace("exec(compile(f.read(), version_file, 'exec'))",
                  "_ns = {}\n        exec(compile(f.read(), version_file, 'exec'), _ns)")
src = src.replace("return locals()['__version__']", "return _ns['__version__']")
assert src != before, 'setup.py did not match the expected pattern — inspect it manually'
sp.write_text(src)
print('patched', sp)

r = subprocess.run([sys.executable, '-m', 'pip', 'install', '-q', str(pkg), '--no-deps'],
                   capture_output=True, text=True)
print(r.stdout[-2000:], r.stderr[-2000:])
r.check_returncode()

# --- torchvision >= 0.17 moved functional_tensor; basicsr still imports the old path ---
# Find the package WITHOUT importing it: importing basicsr executes the very line that
# is broken, so `import basicsr` cannot come before the patch.
import importlib.util
spec = importlib.util.find_spec('basicsr')
assert spec and spec.origin, 'basicsr is not importable — the install above failed'
pkg_dir = pathlib.Path(spec.origin).parent
hit = []
for f in pkg_dir.rglob('*.py'):          # degradations.py is the known one; catch any other
    t = f.read_text()
    if 'functional_tensor' in t:
        f.write_text(t.replace('torchvision.transforms.functional_tensor',
                               'torchvision.transforms.functional'))
        hit.append(f.name)
print('patched:', hit)

# --- realesrgan/version.py is generated by setup.py, which we deliberately skip ---
# realesrgan/__init__.py does `from .version import *`, so the package needs one.
ver = pathlib.Path('realesrgan/version.py')
if not ver.exists():
    v = pathlib.Path('VERSION').read_text().strip() if pathlib.Path('VERSION').exists() else '0.3.0'
    ver.write_text(f"__version__ = '{v}'\n__gitsha__ = 'unknown'\n"
                   f"version_info = tuple(int(x) for x in '{v}'.split('.')[:3])\n")
    print('wrote', ver, v)

# --- register our degradation models with BasicSR's registry ---
# `python realesrgan/train.py` puts realesrgan/ first on sys.path, and that directory
# has its own `archs` package — a top-level `archs` of ours would be shadowed by it
# (ModuleNotFoundError: No module named 'archs.thermal_degradation'). Hence thermal_arch,
# plus an explicit repo-root insert so the import does not depend on PYTHONPATH.
p = pathlib.Path('realesrgan/train.py')
s = p.read_text()
if 'thermal_arch' not in s:
    s = s.replace("import os.path as osp",
                  "import os.path as osp\n"
                  "import sys as _sys\n"
                  "_sys.path.insert(0, osp.dirname(osp.dirname(osp.abspath(__file__))))\n"
                  "import thermal_arch.thermal_degradation  # noqa: F401", 1)
    p.write_text(s)
print(pathlib.Path('realesrgan/train.py').read_text().splitlines()[:8])

# --- verify before moving on ----------------------------------------------
import basicsr
from basicsr.data.degradations import circular_lowpass_kernel  # noqa: F401
from basicsr.utils.registry import MODEL_REGISTRY
import thermal_arch.thermal_degradation  # noqa: F401
import torch
print('basicsr', basicsr.__version__, '| torch', torch.__version__,
      '| cuda', torch.cuda.is_available())
print('registered:', [k for k in MODEL_REGISTRY._obj_map if 'Thermal' in k])


## 3. Build the HR training set

Cuts 480×480 crops, percentile-stretches each image the way the viewer maps °C to the palette,
and drops flat tiles. Aim for **≥ 20k crops**; under ~5k the GAN stage overfits.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python scripts/prepare_thermal_dataset.py \
    --raw /content/drive/MyDrive/thermal_sr/raw \
    --out datasets/thermal_hr \
    --meta datasets/meta_info/thermal_hr.txt \
    --crop 480 --stride 360


## 4. Measure your sensor's fixed-pattern noise (optional, recommended)

Skip if you have no capture — the defaults were measured on a MAG160Core. If you do have one,
paste the printed numbers into the config cell below.


In [ ]:
import pathlib
cap = list((ROOT / 'captures').glob('*.npy'))
if cap:
    !PYTHONPATH=/content/Real-ESRGAN python scripts/estimate_fpn_stats.py {cap[0]}
else:
    print('no captures/*.npy — using the default FPN amplitudes')


## 5. Colab-tuned configs

The repo ymls target a 24 GB card. These overrides fit a T4 (16 GB) and a Colab session:
`gt_size` 256→192, batch 12→8, stage 1 10k→8k iter, stage 2 100k→40k iter.
40k with the pretrained init is enough to see the real behaviour; push further only if the
validation numbers are still improving.


In [ ]:
import yaml, pathlib

# --- paste measured FPN here if you ran cell 4 -----------------------------
FPN = dict(
    fpn_col_sigma=[0.0, 0.010],
    fpn_row_sigma=[0.0, 0.004],
    fpn_map_sigma=[0.0, 0.006],
    fpn_gain_sigma=[0.0, 0.004],
)
STAGE1_ITER, STAGE2_ITER = 8000, 40000
BATCH, GT = 8, 192
# ---------------------------------------------------------------------------

def tune(src, dst, total_iter, milestones):
    o = yaml.safe_load(open(src))
    o.update(FPN)
    o['gt_size'] = GT
    o['queue_size'] = 120
    o['datasets']['train']['gt_size'] = GT
    o['datasets']['train']['batch_size_per_gpu'] = BATCH
    o['datasets']['train']['num_worker_per_gpu'] = 2
    o['train']['total_iter'] = total_iter
    o['train']['scheduler']['milestones'] = milestones
    o['logger']['save_checkpoint_freq'] = 2000
    yaml.safe_dump(o, open(dst, 'w'), sort_keys=False)
    return dst

tune('options/01_thermal_srvgg_x4_net.yml', 'options/colab_01_net.yml', STAGE1_ITER, [int(STAGE1_ITER*0.6)])
o2 = yaml.safe_load(open('options/02_thermal_srvgg_x4_gan.yml'))
o2['path']['pretrain_network_g'] = 'PLACEHOLDER — the stage 2 cell fills this in'
yaml.safe_dump(o2, open('options/_02_tmp.yml', 'w'), sort_keys=False)
tune('options/_02_tmp.yml', 'options/colab_02_gan.yml', STAGE2_ITER, [int(STAGE2_ITER*0.5), int(STAGE2_ITER*0.8)])

# keep checkpoints on Drive so a disconnect cannot lose them
!rm -rf experiments && ln -sfn /content/drive/MyDrive/thermal_sr/experiments experiments
!mkdir -p experiments/pretrained_models
!wget -nc -q -P experiments/pretrained_models https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.5.0/realesr-general-x4v3.pth
!wget -nc -q -P experiments/pretrained_models https://github.com/xinntao/Real-ESRGAN/releases/download/v0.2.2.4/RealESRGAN_x4plus_netD.pth
!ls -la experiments/pretrained_models


## 6. Stage 1 — L1 warm-up (~1 h on a T4)

No adversarial loss yet: the network first learns to undo the thermal degradation. Re-run this
cell after a disconnect; `--auto_resume` picks up the last checkpoint.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py -opt options/colab_01_net.yml --auto_resume


In [ ]:
!ls -la experiments/thermal_srvgg_x4_net/models/ || echo 'stage 1 produced no checkpoint'


## 7. Stage 2 — + perceptual + GAN (~4–6 h on a T4)

Perceptual 0.5 and GAN 5e-2, both below stock Real-ESRGAN: a thermal frame has no texture to
invent, and the GAN term is exactly what invents it.


In [ ]:
# Stage 2 starts from whatever stage 1 actually produced — a Colab disconnect can leave
# the last checkpoint short of the configured total, and a hardcoded iteration number
# then fails with FileNotFoundError.
import pathlib, re, yaml

ck_dir = pathlib.Path('experiments/thermal_srvgg_x4_net/models')
cks = sorted(ck_dir.glob('net_g_*.pth'), key=lambda q: int(re.findall(r'\d+', q.name)[-1])) \
      if ck_dir.is_dir() else []
if not cks:
    raise SystemExit('Stage 1 has produced no checkpoint yet — run the previous cell '
                     'to completion first (it writes into ' + str(ck_dir) + ').')

o = yaml.safe_load(open('options/colab_02_gan.yml'))
o['path']['pretrain_network_g'] = str(cks[-1])
yaml.safe_dump(o, open('options/colab_02_gan.yml', 'w'), sort_keys=False)
print('stage 1 checkpoints:', [c.name for c in cks])
print('stage 2 starts from:', cks[-1])


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python realesrgan/train.py -opt options/colab_02_gan.yml --auto_resume


## 8. Validate — fake hot spots, energy preservation

Run this on every checkpoint you care about. Pick the one where sharpness has stopped improving
but **invented peaks have not started climbing** — usually well before the last iteration.


In [ ]:
ITER = 40000  # checkpoint to check
CKPT = f'experiments/thermal_srvgg_x4_gan/models/net_g_{ITER}.pth'

import pathlib
val = ROOT / 'val_mag160'
if not any(val.iterdir()):
    # no real captures yet: make a stand-in val set by 4x-downsampling held-out HR crops
    import cv2, numpy as np
    src = sorted(pathlib.Path('datasets/thermal_hr').glob('*.png'))[-20:]
    for s in src:
        g = cv2.imread(str(s), cv2.IMREAD_GRAYSCALE)
        cv2.imwrite(str(val / s.name), cv2.resize(g, (160, 120), interpolation=cv2.INTER_AREA))
    print('made a synthetic val set — replace it with real captures when you can')

!PYTHONPATH=/content/Real-ESRGAN python scripts/hallucination_check.py --ckpt {CKPT} --arch srvgg \
    --lr {val} --out results/val_{ITER}

from IPython.display import Image, display
for p in sorted(pathlib.Path(f'results/val_{ITER}').glob('*_check.png'))[:4]:
    display(Image(str(p)))


## 9. Export → ncnn fp16, verify, and pull the model back

Fixed 160×120 input (a static graph converts most cleanly). The second export at 120×160 is for
the rotated orientation the app uses in portrait — build both.


In [ ]:
!PYTHONPATH=/content/Real-ESRGAN python scripts/export_onnx.py --ckpt {CKPT} --arch srvgg --size 160x120 --out export/thermal_x4_160x120
!PYTHONPATH=/content/Real-ESRGAN python scripts/export_onnx.py --ckpt {CKPT} --arch srvgg --size 120x160 --out export/thermal_x4_120x160

!chmod +x scripts/convert_ncnn.sh
!./scripts/convert_ncnn.sh export/thermal_x4_160x120 160 120
!mv export/thermal_x4_fp16.param export/thermal_160x120_fp16.param
!mv export/thermal_x4_fp16.bin   export/thermal_160x120_fp16.bin
!./scripts/convert_ncnn.sh export/thermal_x4_120x160 120 160
!mv export/thermal_x4_fp16.param export/thermal_120x160_fp16.param
!mv export/thermal_x4_fp16.bin   export/thermal_120x160_fp16.bin


In [ ]:
# PyTorch vs ncnn — fp16 should land under ~6e-3 max, 1e-3 mean (see the notes in the script)
!PYTHONPATH=/content/Real-ESRGAN python scripts/verify_ncnn.py --ref export/thermal_x4_160x120_ref.pt \
    --param export/thermal_160x120_fp16.param --bin export/thermal_160x120_fp16.bin


In [ ]:
# copy the deployable files to Drive, and offer a direct download
!mkdir -p /content/drive/MyDrive/thermal_sr/export
!cp export/thermal_*_fp16.* {CKPT} /content/drive/MyDrive/thermal_sr/export/
!cd export && zip -q -r /content/thermal_ncnn.zip thermal_*_fp16.param thermal_*_fp16.bin
from google.colab import files
files.download('/content/thermal_ncnn.zip')


---
### What to send back

`thermal_ncnn.zip` (the four `.param`/`.bin` files) plus the `results/val_*/summary.csv` numbers.
The `.pth` checkpoint on Drive is worth keeping too — re-converting is free, re-training is not.
